# EDA des Behavior Trees XML

Ce notebook construit un pipeline d'analyse complet pour le dataset BTGenBot/Nav4Rails. Chaque entrée contient une consigne (`instruction`), une description en langage naturel (`input`) et un Behavior Tree XML (`output`).

Objectifs principaux :

1. Charger et auditer la qualité du dataset.
2. Parser les Behavior Trees XML avec `xml.etree.ElementTree`.
3. Extraire des métriques structurelles, syntaxiques et sémantiques.
4. Construire des tables `pandas` exploitables pour l'analyse.
5. Visualiser les distributions, motifs, attributs et graphes BT.
6. Explorer la relation entre texte naturel et structure XML.
7. Préparer les pistes avancées : Tree Edit Distance, clustering, embeddings de graphes, GNN/Tree-LSTM et future conversion JSONL.

Le dépôt originel BTGenBot est ajouté comme sous-module dans `external/BTGenBot` pour garder une référence traçable au projet publié : https://github.com/AIRLab-POLIMI/BTGenBot.git

In [ ]:
from __future__ import annotations

import json
import math
import re
import statistics
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import networkx as nx
except ImportError:
    nx = None

try:
    from sklearn.cluster import KMeans
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.preprocessing import StandardScaler
except ImportError:
    KMeans = None
    TfidfVectorizer = None
    cosine_similarity = None
    StandardScaler = None

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET_PATH = PROJECT_ROOT / "dataset" / "bt_dataset.json"
SUBMODULE_PATH = PROJECT_ROOT / "external" / "BTGenBot"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET_PATH}")
print(f"BTGenBot submodule present: {SUBMODULE_PATH.exists()}")

## Fonctions utilitaires

Les fonctions ci-dessous gardent volontairement les transformations explicites : parsing XML strict, nettoyage minimal optionnel, parcours d'arbre, extraction de métriques, construction de tables et conversion en graphe. Le dataset contient plusieurs dialectes de Behavior Trees, donc les règles de catégorisation restent descriptives plutôt que normatives.